FRAME TO FRAME MODEL INFERENCE & EVALUATION


In [11]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import warnings
warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

sys.path.append('/home/adelval/BTS/TFM/test/src/net')
sys.path.append('/home/adelval/BTS/TFM/test/src/train')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration

In [12]:
x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
print('  x_test: %s' % len(x_test))
print(x_test)

  x_test: 1
['/home/adelval/BTS/TFM/audios/audio_1.wav']


In [13]:
import soundfile as sf
import numpy as np
def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

In [14]:
audio, fs = read_audio(x_test[0])
print(audio)
print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

frame_size = 0.01 # 10 ms
frame_samples = int(frame_size * fs)
print(f'Un frame tiene una duración de {frame_samples} samples')


# Simulates Incoming Buffer 
shift_size = 0.01 # 10 ms
shift_samples = int(shift_size * fs)
window_size = 0.04 # 40ms
window_samples = int(window_size * fs)
print(f'La ventana tiene una duración de {window_samples} samples')
frame_received = []
it = 0
# First buffer need to have 4 frames
while(len(frame_received) < window_samples):
    frame_received = np.concatenate([frame_received,audio[it*shift_samples:it*shift_samples+frame_samples]])
    it+=1
    print(len(frame_received))
print(frame_received)

[  0  -1  -1 ... -30 -34 -30]
La duración del audio es 4.655 segundos y 74480 muestras
Un frame tiene una duración de 160 samples
La ventana tiene una duración de 640 samples
160
320
480
640
[ 0. -1. -1.  0. -1.  0.  0.  0.  1.  1.  0.  0.  0. -1.  1.  0.  0.  0.
  0.  1.  0.  1.  0.  0.  1.  1.  0.  0.  0.  0.  0. -1.  0.  0.  0. -1.
  0.  0.  1.  0.  0.  0.  0.  1.  0.  1.  0.  0.  0.  0. -1.  0.  0.  0.
  0.  0. -1.  0.  0.  0.  1.  1.  0.  0. -1.  0.  0.  1.  0.  0. -1.  1.
  0.  0.  0.  0. -1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  1.  0.
  0.  0. -1.  0.  1.  0.  0.  1. -1.  0. -1. -1.  0.  0.  0.  0.  0.  0.
  0.  0. -1.  0. -1.  1.  0.  1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  1.  0. -1.  0.  0.  0.  0.  0.  0. -1.  0.  0. -1.
  0.  0.  0.  0.  0.  0.  0.  1.  1.  0.  0.  0.  0.  0.  0.  0. -1.  0.
  0.  0.  0.  0.  0. -1. -1.  0.  0.  0.  0.  0.  1.  0.  1. -1.  0.  0.
 -1.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  0.  1.  0.  1.  0.  0.  0.
 -1. -

In [15]:
## PROCESADO INFERENCIA

from spicy import signal

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010):
    N = int(Ns * fs)            # Number of samples in each window
    M = int(Ms * fs)            # Step size (number of samples between window starts)
    n = (len(x) + M - 1) // M   # Number of frames
    print(len(x))
    print("Number of shifts", n)    
    T = (n - 1) * M + N         # Total signal length needed to fit the frames
    print(T)
    xa = x.copy()
    if T > len(x):
        xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, n * M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    return xa[ind.astype(int).T].astype(np.float32)

def hamming(X):
    w = np.hamming(len(X))
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[0])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=0)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]

N =[int(wi * fs) for wi in w]
F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
#fb = [fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]

x = offset(frame_received)

# Emphasis to increase the amplitude of high freq
x = preemphasis(x)

XX = []

X = hamming(x)
print("size de X ",X.shape)
# Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
Xfft = fft(X, nfft[0])
print(f"la shape de Xfft es {Xfft.shape}")
X = Xfft * Xfft # Power spectrum
X = np.asarray(X, dtype=np.float32)

frame_psd = X
print("Vector con Power Spectral Density del frame  \n", frame_psd)


size de X  (640,)
la shape de Xfft es (512,)
Vector con Power Spectral Density del frame  
 [3.35846678e-03 3.01946755e-02 8.89347121e-02 1.43956587e-01
 1.95857048e-01 2.08799928e-01 9.91671979e-02 3.25955562e-02
 5.72605394e-02 7.49174133e-02 6.86633140e-02 6.55169636e-02
 2.04117328e-01 1.94240674e-01 2.65824869e-02 6.71783388e-02
 4.92178530e-01 1.13121772e+00 1.80837321e+00 1.83699596e+00
 5.51402390e-01 1.25074372e-01 8.43026996e-01 1.71314394e+00
 2.49174166e+00 1.44956577e+00 1.07196383e-01 2.08683777e+00
 4.48774004e+00 1.80508876e+00 1.64744318e-01 1.44931662e+00
 1.10694563e+00 1.35797963e-01 8.24924707e-01 2.43872094e+00
 1.75579381e+00 2.74597466e-01 2.15600073e-01 1.76141202e-01
 4.85421658e-01 7.00046957e-01 9.40734386e-01 4.16991591e-01
 1.37973475e+00 7.64781356e-01 1.38883996e+00 8.20622826e+00
 1.94445114e+01 2.09126167e+01 1.18048201e+01 6.94107533e+00
 1.15719070e+01 1.99116554e+01 2.12112560e+01 1.16127071e+01
 3.32672501e+00 2.70395279e-01 3.31343555e+00 7.788237

In [16]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients

def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b


fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
N =[int(wi * fs) for wi in w]
F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
print(f'fb es {len(fb[0])}')
dct = [ f_base_dct(Bi) for Bi in B] 
print(f'dct es {len(dct[0])}')


x = frame_received
x = offset(x)
x = preemphasis(x)
XX = []
for i,w in enumerate(w):
    X = hamming(x)
    
    Xfft = fft(X, nfft[i])
    Xb = np.log(Xfft.dot( fb[i] ) + 1)
    print(Xb.shape)
    Xc = Xb.dot(dct[i])            
    print(Xc.shape)                    
    
    X = np.concatenate( [Xb, Xc], 0 )
    
    X = np.asarray(X, dtype=np.float32)
    print(X.shape)
    XX.append(X)
XX = np.concatenate(XX, 0)
  

frame_fbmfcc = XX
print("El tamaño de x08k2 es: ",frame_fbmfcc.shape)
print(frame_fbmfcc)

fb es 512
dct es 32
(32,)
(32,)
(64,)
El tamaño de x08k2 es:  (64,)
[ 2.3243648e-01  2.8519303e-01  5.6733555e-01  6.8346107e-01
  7.8756684e-01  7.0884502e-01  5.9491789e-01  8.4041601e-01
  1.4721725e+00  1.1784921e+00  9.0479803e-01  1.6591580e+00
  1.6208335e+00  1.6077632e+00  1.6622918e+00  1.8071356e+00
  1.8437001e+00  2.0725408e+00  1.9555092e+00  2.0416002e+00
  2.2954106e+00  2.2919426e+00  2.3828988e+00  2.2032447e+00
  2.2712402e+00  2.2792289e+00  2.4267817e+00  2.4498546e+00
  2.5948927e+00  2.6776021e+00  2.5291076e+00  2.5170765e+00
  9.4479094e+00 -4.0548253e+00 -7.7350938e-01 -3.0885884e-01
 -1.7371446e-02 -1.3842647e-01  2.2886336e-02 -9.3668468e-02
 -2.3326437e-01  3.2749102e-02 -2.6294720e-01 -1.0510495e-01
 -1.6771248e-01 -1.5973203e-01  5.9122931e-02  3.8319921e-01
  2.1123955e-01  1.1837896e-02 -1.7019042e-01 -2.6861385e-01
 -1.1057513e-01  1.0392741e-01  3.1165832e-01  8.4828727e-02
 -5.4433260e-02 -9.2765108e-02  6.0782689e-03 -4.7130480e-02
  1.2586702e-01  

In [17]:
# Log scale for PSD
scale=1.
eps=1e-8
x = frame_psd    
x = np.abs(x)
x = scale * np.log10(x + eps)      

frame_psd_log  = x   
print("El tamaño de x08k en log es: ",frame_psd_log.shape)
print(frame_psd[:10])
print(frame_psd_log[:10])

El tamaño de x08k en log es:  (512,)
[0.00335847 0.03019468 0.08893471 0.14395659 0.19585705 0.20879993
 0.0991672  0.03259556 0.05726054 0.07491741]
[-2.4738579  -1.5200696  -1.0509287  -0.84176844 -0.7080608  -0.68026966
 -1.003632   -1.4868416  -1.2421446  -1.1254172 ]


In [18]:
# Normalization of fbmfcc
import pickle

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

file = '/home/adelval/BTS/TFM/test/data/model/fe1_norm1.pkl'  # de donde salen??
mu, std = read_pkl(file)
x = frame_fbmfcc
print(np.mean(frame_fbmfcc))
x -= mu
x /= std + 1e-6
frame_fbmfcc_norm = x 

print("El tamaño de x08k2 en log es: ",frame_fbmfcc_norm.shape)
print(np.mean(frame_fbmfcc_norm))
print((frame_fbmfcc_norm))

0.8939117
El tamaño de x08k2 en log es:  (64,)
-1.2592797
[-2.984838   -2.8981009  -2.7997372  -2.7685244  -2.7185802  -2.7820437
 -2.8423624  -2.746138   -2.5031915  -2.696202   -2.8533165  -2.4881728
 -2.5689733  -2.6227412  -2.606246   -2.5584383  -2.5598524  -2.4504247
 -2.5063837  -2.4680147  -2.3540194  -2.3565807  -2.3064573  -2.3889208
 -2.3457603  -2.2829013  -2.1568913  -2.0873945  -1.9831898  -1.9313042
 -1.9824072  -1.8421077  -2.9405684  -1.1840508   0.01492556 -0.5953394
  0.33280596  0.14901951  0.46964574  0.2581699  -0.04858989  0.07717898
 -0.25317582 -0.15703724 -0.04886905 -0.38283515  0.31059605  0.8837946
  0.7326705   0.13174751 -0.07904613 -0.45833227 -0.03709359  0.30816188
  0.9164052   0.2586629  -0.03898656 -0.204009    0.11430339 -0.13102315
  0.44306713  0.44455576 -0.27737486 -0.16306686]


In [19]:
# Concatenación de x08k y x08k2

    
frame_concat = np.concatenate( (frame_psd_log,frame_fbmfcc_norm), 0 )
print("El tamaño de x08k tras la concatenacion es: ",frame_concat.shape)
print(f'de fft {frame_psd_log[1:2]} y de fb {frame_fbmfcc_norm[1:2]} y de concat {frame_concat[1:2]} y {frame_concat[513:514]}')

El tamaño de x08k tras la concatenacion es:  (576,)
de fft [-1.5200696] y de fb [-2.8981009] y de concat [-1.5200696] y [-2.8981009]


In [20]:
import os
import gzip
import pickle
def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

input_dim, output_dim = load_obj('/home/adelval/BTS/TFM/test/data/model/dimensions.pkl') 

print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))

  input_dim: 576
  output_dim: 512


In [21]:
import sys
sys.path.append('/home/adelval/BTS/TFM/test/src/net')

from net_snr import Net_snr

net_snr = Net_snr(input_dim, output_dim, cuda=False)
net_snr.load_theta('/home/adelval/BTS/TFM/test/data/model/theta_last')


  Net_snr:
    nb_params: 29.99M
    cuda: False
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    reading obj /home/adelval/BTS/TFM/test/data/model/theta_last


In [22]:
# Model inference
import scipy.io
import matplotlib.pyplot as plt

net_snr.set_mode_train(False)

f = '/home/adelval/BTS/TFM/audios/frame_test.mat'
print(f)
x = frame_concat
print(frame_concat)
print(x.shape)

# Test
x = x.reshape(1, -1)
print(x.shape)
#for i in range(1) :
#    x = np.concatenate([x,x], 0)
#print(x.shape)
    
print(' Checking: ' + f)
#if not os.path.exists(f):
print('Processing: ' + f)
snr = net_snr.predict(x)
print(snr[0,:10])
snr = to_numpy(snr.squeeze())
print(snr.shape)
scipy.io.savemat(f, mdict={'snr': snr})
x = to_numpy(x.squeeze())
# snr = to_numpy(snr.squeeze())



/home/adelval/BTS/TFM/audios/frame_test.mat
[-2.4738579  -1.5200696  -1.0509287  -0.84176844 -0.7080608  -0.68026966
 -1.003632   -1.4868416  -1.2421446  -1.1254172  -1.1632752  -1.1836462
 -0.6901201  -0.7116598  -1.5754043  -1.1727707  -0.30787736  0.0535462
  0.25728807  0.2641082  -0.25853136 -0.9028316  -0.07415852  0.23379387
  0.39650303  0.16123793 -0.96981984  0.3194887   0.6520277   0.25649858
 -0.78318954  0.16116327  0.04412629 -0.86710674 -0.08358569  0.38716212
  0.24447353 -0.5613035  -0.6663511  -0.754139   -0.31388086 -0.15487283
 -0.02653298 -0.3798727   0.13979562 -0.11646271  0.1426522   0.9141436
  1.2887971   1.3204085   1.0720594   0.8414268   1.0634049   1.2991074
  1.3265665   1.0649335   0.52201694 -0.5680009   0.5202786   0.8914392
  0.8193027   0.61953    -0.15539357 -0.7660339  -0.5859978  -1.0000855
 -0.20170373  0.63671786  0.54845613  0.27862656  0.89586496  0.75610214
  0.9643185   1.304961    1.2650245   1.2249322   1.4048554   1.5669436
  1.7221236   

In [136]:
hello = torch.tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])

std_hello = hello.std(dim=1,keepdim = True) +1 
print(std_hello)

tensor([[nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan]])


In [137]:
print(f'La SNR del frame resulta de {snr.shape} y {snr}')

La SNR del frame resulta de (512,) y [0.31268036 0.4395056  0.58460456 0.54150844 0.5077111  0.41632956
 0.36945698 0.35867354 0.43146437 0.495409   0.4340879  0.4385882
 0.48164198 0.5016437  0.60070926 0.6710255  0.6858011  0.6504705
 0.587875   0.6104359  0.6617553  0.61710835 0.50564754 0.43527114
 0.40025038 0.43675762 0.4684086  0.43326354 0.3183867  0.30260986
 0.31289294 0.3143674  0.33864754 0.42000538 0.4416301  0.461888
 0.395506   0.33415905 0.37492323 0.3833868  0.3595675  0.35113138
 0.35045877 0.38161564 0.45895264 0.47859585 0.49895066 0.49997938
 0.5558083  0.5551234  0.5619291  0.59301305 0.6417939  0.70637107
 0.76669854 0.76615834 0.76675683 0.77169216 0.7713935  0.7672087
 0.7571141  0.75153995 0.7538289  0.78414905 0.81168365 0.8283055
 0.7872705  0.7737678  0.82246643 0.8416245  0.82184976 0.80067974
 0.7862123  0.7987883  0.8123096  0.79199994 0.8072029  0.83175826
 0.865449   0.8772224  0.8932048  0.8821673  0.8711683  0.8001311
 0.7512971  0.73706853 0.8264855

EVALUATION

In [138]:
# Real evaluation

from __future__ import print_function
from __future__ import division
import time, os, sys
import warnings
warnings.simplefilter('ignore')

sys.path.append('/home/adelval/BTS/TFM/test/src/train')
sys.path.append('/home/adelval/BTS/TFM/test/src/eval')

import numpy as np
from eval_utils import *
from vvtk_net.config import Configuration
from scipy.io import wavfile
from scipy.io import loadmat

In [139]:
# Aplicar mascara calculada con el frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    it = int(np.floor((data.size-frame)/shift))
    it = 1
    print(f'La ventana se desplazara {it} veces')
    
    for i in range(0,it):
        print(f'El frame sin enventanado resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        print(f'El tamaño de la salida del filtro sera {outf.shape}')
        print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        print(f'El frame mejorado resulta {yw[:10]}')
    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    yw = apply_filter(data, filt, frame, shift, nfft)
    

    return yw, filt


fs=16000
B=[32]
w=[0.040]
frame = int(fs*w[0])
m=0.010
shift = int(fs*m)
nfft=1024
gmin = 0.0562

wav = x_test[0]
print(f'Audio seleccionado es --> {wav}')

x = np.array(frame_received, dtype=np.float32) / 2 ** 15      # 0.04 * fs = 640 samples 
print(f'El frame sin enventanado resulta {x[:10]}')
snr_net_file = '/home/adelval/BTS/TFM/audios/frame_test.mat'
snr_net = loadmat(snr_net_file)['snr'].T
print(f'La snr cargada es de dimensiones {snr_net.shape}')
print(f'La mascara del primer fragmento es {snr_net[:,0]}')

    
xenh, filt = noiseReduction(x, snr_net, fs, frame, shift, nfft, gmin)
print(f'Las dimensiones del filtro son {filt.shape}')
print(f'El frame mejorado es {xenh[:100]}')



Audio seleccionado es --> /home/adelval/BTS/TFM/audios/audio_1.wav
El frame sin enventanado resulta [ 0.0000000e+00 -3.0517578e-05 -3.0517578e-05  0.0000000e+00
 -3.0517578e-05  0.0000000e+00  0.0000000e+00  0.0000000e+00
  3.0517578e-05  3.0517578e-05]
La snr cargada es de dimensiones (512, 1)
La mascara del primer fragmento es [0.31268036 0.4395056  0.58460456 0.54150844 0.5077111  0.41632956
 0.36945698 0.35867354 0.43146437 0.495409   0.4340879  0.4385882
 0.48164198 0.5016437  0.60070926 0.6710255  0.6858011  0.6504705
 0.587875   0.6104359  0.6617553  0.61710835 0.50564754 0.43527114
 0.40025038 0.43675762 0.4684086  0.43326354 0.3183867  0.30260986
 0.31289294 0.3143674  0.33864754 0.42000538 0.4416301  0.461888
 0.395506   0.33415905 0.37492323 0.3833868  0.3595675  0.35113138
 0.35045877 0.38161564 0.45895264 0.47859585 0.49895066 0.49997938
 0.5558083  0.5551234  0.5619291  0.59301305 0.6417939  0.70637107
 0.76669854 0.76615834 0.76675683 0.77169216 0.7713935  0.7672087
 0.7

In [107]:
# Aplicar mascara calculada con el audio completo

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    it = int(np.floor((data.size-frame)/shift))
    it = 1
    print(f'La ventana se desplazara {it} veces')
    
    for i in range(0,it):
        print(f'El frame sin enventanado resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        print(Xfft.shape)
        print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        print(f'El tamaño de la salida del filtro sera {outf.shape}')
        print(f'La salida del filtro es {outf[:10]}')

        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        print(f'El frame mejorado resulta sin OLA es {outw[:10]}')

        #print(f'La salida de la ifft tendra {outw.shape} samples') # de las que nos quedamos con 640 porque el resto son relleno
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    print(snr_net.shape)
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


fs=16000
B=[32]
w=[0.040]
frame = int(fs*w[0])
m=0.010
shift = int(fs*m)
nfft=1024
gmin = 0.0562

wav = x_test[0]
print(f'Audio seleccionado es --> {wav}')

x = np.array(frame_received, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples
print(f'El frame sin enventanado resulta {x[:10]}')
snr_net_file = '/home/adelval/BTS/TFM/audios/audio_1.mat'
snr_net = loadmat(snr_net_file)['snr'].T
snr_net_cropped = snr_net[:,0]
snr_net_cropped = snr_net_cropped[:, np.newaxis]

print(f'La snr cargada es de dimensiones {snr_net_cropped.shape}')
print(f'La máscara del primer fragmento es {snr_net_cropped}')
    
xenh, filt = noiseReduction(x, snr_net_cropped, fs, frame, shift, nfft, gmin)
print(f'Las dimensiones del filtro son {filt.shape}')

print(f'El frame mejorado es {xenh[:10]}')


Audio seleccionado es --> /home/adelval/BTS/TFM/audios/audio_1.wav
El frame sin enventanado resulta [ 0.0000000e+00 -3.0517578e-05 -3.0517578e-05  0.0000000e+00
 -3.0517578e-05  0.0000000e+00  0.0000000e+00  0.0000000e+00
  3.0517578e-05  3.0517578e-05]
La snr cargada es de dimensiones (512, 1)
La máscara del primer fragmento es [[1.0000000e+00]
 [9.9997437e-01]
 [9.9992537e-01]
 [9.9995816e-01]
 [9.9863005e-01]
 [7.3849213e-01]
 [5.2971745e-01]
 [9.2125833e-01]
 [9.9565923e-01]
 [8.8152605e-01]
 [6.4611769e-01]
 [1.2178899e-02]
 [5.5099939e-05]
 [4.4791112e-03]
 [2.8145444e-01]
 [2.8170618e-01]
 [6.0391074e-01]
 [1.1674175e-01]
 [5.1899222e-03]
 [2.0049827e-04]
 [4.9005379e-05]
 [2.7175221e-04]
 [1.5303500e-04]
 [6.2884639e-05]
 [4.3917134e-05]
 [2.2219743e-05]
 [4.7926210e-06]
 [2.8836922e-07]
 [2.9348135e-07]
 [2.1066191e-06]
 [6.6840061e-05]
 [2.5349329e-04]
 [1.2558467e-04]
 [1.7133940e-04]
 [4.3591097e-04]
 [2.8011350e-02]
 [6.1377132e-01]
 [9.6601951e-01]
 [9.8784196e-01]
 [9.97

In [18]:
# De lo obtenido me sirven los primeros 10ms, es decir, el primer paquete recibido
# Este lo enviaría al receptor.


frame_to_send = xenh[0:int(frame_size*fs)]